# Step 1 launcher

Rendered and submitted by `tools/kaggle_run.py`; do not edit or run by hand. The
commit and configuration below are pinned at submission time. This notebook only
clones an exact source revision and invokes the repository-owned runner once:
all model, data, training, evaluation, and packaging logic lives in Git.

Ephemeral state stays under `/tmp/step1-runtime`. `/kaggle/working` is output only.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/escher-bach/actuallybuildingstuff.git"
GIT_COMMIT = "__FINAL_COMMIT_SHA__"
CONFIG_REL = "__CONFIG_REL__"

RUNTIME = Path("/tmp/step1-runtime")
WORKING = Path("/kaggle/working")
SOURCE = RUNTIME / "actuallybuildingstuff"
PROJECT = SOURCE / "baby-llm-foundations"
OUTPUT = WORKING / "step1-results"

assert len(GIT_COMMIT) == 40 and all(c in "0123456789abcdef" for c in GIT_COMMIT)
assert not SOURCE.exists(), f"fresh batch session required; already exists: {SOURCE}"
RUNTIME.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
env = os.environ.copy()
env.update({
    "GIT_TERMINAL_PROMPT": "0",
    "PYTHONUNBUFFERED": "1",
    "PIP_DISABLE_PIP_VERSION_CHECK": "1",
    "WANDB_MODE": "disabled",
    "TOKENIZERS_PARALLELISM": "false",
    # Keep every build and dependency cache off the published output tree.
    "CARGO_HOME": str(RUNTIME / "caches" / "cargo"),
    "CARGO_TARGET_DIR": str(RUNTIME / "build" / "cargo-target"),
    "PIP_CACHE_DIR": str(RUNTIME / "caches" / "pip"),
    "HF_HOME": str(RUNTIME / "caches" / "huggingface"),
})

subprocess.run(["git", "clone", REPO_URL, str(SOURCE)], check=True, env=env)
subprocess.run(
    ["git", "-C", str(SOURCE), "checkout", "--detach", GIT_COMMIT],
    check=True,
    env=env,
)
resolved = subprocess.check_output(
    ["git", "-C", str(SOURCE), "rev-parse", "HEAD"],
    text=True,
    env=env,
).strip()
assert resolved == GIT_COMMIT, (resolved, GIT_COMMIT)
assert (PROJECT / CONFIG_REL).is_file(), PROJECT / CONFIG_REL

In [ ]:
cmd = [
    sys.executable,
    "-m",
    "step1_experiments.runner",
    "--config", str(PROJECT / CONFIG_REL),
    "--output-root", str(OUTPUT),
    "--resume", "auto",
]

completed = subprocess.run(
    cmd,
    cwd=str(PROJECT / "step1" / "python"),
    env=env,
    check=False,
)
if completed.returncode != 0:
    raise RuntimeError(
        f"Step 1 runner failed with exit code {completed.returncode}; "
        f"collect the analysis payload from {OUTPUT}"
    )